# Public Dataset -> Local Lake Pipeline

Ноутбук побудований як покроковий runbook: `download -> write -> read`.

Орієнтир: команда Java/Scala engineering. У кожному блоці нижче є короткі підказки, чим ці формати зазвичай читають у JVM-екосистемі.


## 0) Install (одноразово)

Що робить крок:
- ставить локальний пакет у editable-режимі.

Для Java-читачів:
- це Python runtime частина для підготовки data lake; читання даних може виконуватись вже JVM-інструментами.


In [1]:
# Якщо залежності вже встановлені, пропусти
# %pip install -e ..


## 1) Imports

Що робить крок:
- підключає модулі для download/write/read.

Формати в цьому ноутбуку:
- `Parquet`: базовий колонковий формат файлів.
- `Delta`: table format поверх Parquet + ACID/лог транзакцій.
- `Iceberg`: table format з незалежним metadata layer для multi-engine читання.


In [2]:
from pathlib import Path
import json
import tempfile
import sys

import pandas as pd

ROOT = Path.cwd().parent
sys.path.append(str(ROOT / "src"))

from kaggle_s3_lake.config import Settings
from kaggle_s3_lake.kaggle_download import download_file_from_url, load_dataframe
from kaggle_s3_lake.writers import write_parquet, write_delta, write_iceberg, dataframe_schema
from kaggle_s3_lake.readers import read_parquet, read_delta, read_iceberg


## 2) Parameters

Що робить крок:
- задає URL датасету, локальні шляхи, namespace/table для Iceberg.

Що зазвичай хочуть Java-команди:
- стабільний `DATASET_ID` для predictable path/table naming;
- окремий `LAKE_ROOT` для dev/stage/prod середовищ.


In [3]:
SOURCE_URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
DATASET_ID = "titanic"
LAKE_ROOT = "./data_lake"
PREFIX = "kaggle-datasets"
NAMESPACE = "kaggle"
CATALOG_NAME = "local"
TABLE_NAME = DATASET_ID.replace("/", "_").replace("-", "_").lower()
SAMPLE_LIMIT = None    # приклад: 1000


## 3) Runtime setup

Що робить крок:
- обчислює локальні шляхи для артефактів;
- створює директорії під Parquet/Delta/Iceberg/manifest.

Порада для JVM deployment:
- збережіть структуру шляху стабільною, щоб Spark/Flink jobs не залежали від ad-hoc naming.


In [4]:
settings = Settings.from_env()
lake_root = Path(LAKE_ROOT).resolve()
dataset_key = DATASET_ID.replace("/", "__")
dataset_root = (lake_root / PREFIX / dataset_key).resolve()

parquet_uri = str((dataset_root / "parquet" / "data.parquet").resolve())
delta_uri = str((dataset_root / "delta").resolve())
warehouse_uri = str((lake_root / PREFIX / "iceberg").resolve())
manifest_path = (dataset_root / "manifest.json").resolve()

Path(parquet_uri).parent.mkdir(parents=True, exist_ok=True)
Path(delta_uri).mkdir(parents=True, exist_ok=True)
Path(warehouse_uri).mkdir(parents=True, exist_ok=True)

print("Parquet:", parquet_uri)
print("Delta:  ", delta_uri)
print("Iceberg warehouse:", warehouse_uri)
print("Manifest:", manifest_path)


Parquet: /Users/dmytrosylenok/PycharmProjects/test-data-protocols/notebooks/data_lake/kaggle-datasets/titanic/parquet/data.parquet
Delta:   /Users/dmytrosylenok/PycharmProjects/test-data-protocols/notebooks/data_lake/kaggle-datasets/titanic/delta
Iceberg warehouse: /Users/dmytrosylenok/PycharmProjects/test-data-protocols/notebooks/data_lake/kaggle-datasets/iceberg
Manifest: /Users/dmytrosylenok/PycharmProjects/test-data-protocols/notebooks/data_lake/kaggle-datasets/titanic/manifest.json


## 4) Download dataset from URL

Що робить крок:
- завантажує публічний CSV;
- читає його в DataFrame.

Чому це зручно:
- не потрібен Kaggle auth;
- demo запускається з коробки.


In [5]:
tmp_dir = tempfile.TemporaryDirectory(prefix="dataset_download_")
downloaded_file = download_file_from_url(SOURCE_URL, Path(tmp_dir.name))
df, selected_file = load_dataframe([downloaded_file], preferred_file=None)

if SAMPLE_LIMIT:
    df = df.head(SAMPLE_LIMIT).copy()

print(f"selected_file={selected_file}")
print(f"rows={len(df)}, cols={len(df.columns)}")


selected_file=/var/folders/r6/wpgf8l695qg1_7yxjlf_40r00000gn/T/dataset_download_jjovu76p/titanic.csv
rows=891, cols=12


In [6]:
display(df.head(10))
display(pd.DataFrame(list(dataframe_schema(df))))


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


,name,dtype
0,PassengerId,int64
1,Survived,int64
2,Pclass,int64
3,Name,str
4,Sex,str
5,Age,float64
6,SibSp,int64
7,Parch,int64
8,Ticket,str
9,Fare,float64


## 5) Write Parquet

Що робить крок:
- записує plain Parquet-файл.

Java/JVM читання (типово):
- Spark SQL (`spark.read.parquet(...)`),
- Flink Table/SQL,
- `parquet-mr`/Hadoop stack для низькорівневого доступу.

Maven (приклад):
- `org.apache.parquet:parquet-hadoop`
- `org.apache.spark:spark-sql_2.12` (або ваша Scala binary версія)


In [7]:
write_parquet(df, parquet_uri)
print(f"Parquet written: {parquet_uri}")


Parquet written: /Users/dmytrosylenok/PycharmProjects/test-data-protocols/notebooks/data_lake/kaggle-datasets/titanic/parquet/data.parquet


## 6) Write Delta

Що робить крок:
- створює Delta table (Parquet + `_delta_log`).

Java/JVM читання (типово):
- Spark + Delta Lake extension,
- Databricks runtime,
- Delta Kernel (для lightweight readers).

Maven (приклад для Spark):
- `io.delta:delta-spark_2.12`

Важливо:
- версії `spark` і `delta-spark` мають бути сумісні.


In [8]:
write_delta(df, delta_uri)
print(f"Delta written: {delta_uri}")


Delta written: /Users/dmytrosylenok/PycharmProjects/test-data-protocols/notebooks/data_lake/kaggle-datasets/titanic/delta


## 7) Write Iceberg

Що робить крок:
- створює Iceberg table у SQL catalog;
- пише data files + metadata в warehouse.

Java/JVM читання (типово):
- Spark + Iceberg runtime,
- Flink + Iceberg,
- Trino/Presto/Athena (через catalog integration).

Maven (приклади):
- `org.apache.iceberg:iceberg-core`
- `org.apache.iceberg:iceberg-spark-runtime-3.5_2.12` (під вашу Spark версію)


In [9]:
iceberg_ref = write_iceberg(
    df,
    catalog_name=CATALOG_NAME,
    namespace=NAMESPACE,
    table_name=TABLE_NAME,
    catalog_uri=settings.iceberg_catalog_uri,
    warehouse_uri=warehouse_uri,
)
print(f"Iceberg written: {iceberg_ref}")


Iceberg written: local.kaggle.titanic


## 8) Write manifest.json

Що робить крок:
- пише єдиний маніфест з шляхами до всіх форматів і schema snapshot.

Для Java-команд:
- `manifest.json` можна використовувати як bootstrap metadata для downstream jobs/services.


In [10]:
manifest = {
    "dataset": DATASET_ID,
    "source_url": SOURCE_URL,
    "source_file": str(selected_file),
    "rows": int(len(df)),
    "schema": list(dataframe_schema(df)),
    "artifacts": {
        "parquet": parquet_uri,
        "delta": delta_uri,
        "iceberg": {
            "catalog": CATALOG_NAME,
            "namespace": NAMESPACE,
            "table": TABLE_NAME,
            "ref": iceberg_ref,
            "catalog_uri": settings.iceberg_catalog_uri,
            "warehouse_uri": warehouse_uri,
        },
    },
}
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Manifest written: {manifest_path}")


Manifest written: /Users/dmytrosylenok/PycharmProjects/test-data-protocols/notebooks/data_lake/kaggle-datasets/titanic/manifest.json


## 9) Read Parquet

Що робить крок:
- валідація, що Parquet читається і повертає очікувані rows/columns.


In [11]:
df_parquet = read_parquet(parquet_uri)
print(df_parquet.shape)
display(df_parquet.head(10))


(891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


## 10) Read Delta

Що робить крок:
- валідація Delta-read після запису.

Для Java Spark-пайплайнів:
- на цьому кроці зазвичай додають smoke-check query по ключових колонках.


In [12]:
df_delta = read_delta(delta_uri)
print(df_delta.shape)
display(df_delta.head(10))


(891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


## 11) Read Iceberg

Що робить крок:
- валідація доступу через catalog + table identifier.

Для Java production usage:
- читати через catalog (`namespace.table`) замість raw file path.


In [13]:
df_iceberg = read_iceberg(
    catalog_name=CATALOG_NAME,
    namespace=NAMESPACE,
    table_name=TABLE_NAME,
    catalog_uri=settings.iceberg_catalog_uri,
    warehouse_uri=warehouse_uri,
)
print(df_iceberg.shape)
display(df_iceberg.head(10))


(891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


## 12) Cleanup (optional)

Що робить крок:
- чистить тимчасову директорію завантаження.

Не видаляє `data_lake` артефакти, щоб їх можна було читати іншими сервісами.


In [14]:
tmp_dir.cleanup()


## JVM Dependency Cheat Sheet

### Parquet
- Spark: `org.apache.spark:spark-sql_2.12`
- Parquet low-level: `org.apache.parquet:parquet-hadoop`

### Delta
- Spark connector: `io.delta:delta-spark_2.12`
- Слідкуйте за matrix сумісності Spark <-> Delta

### Iceberg
- Core API: `org.apache.iceberg:iceberg-core`
- Spark runtime: `org.apache.iceberg:iceberg-spark-runtime-<spark_version>_2.12`
- Для Flink/Trino використовуйте відповідні Iceberg runtime modules
